# Norton: classification preview

Run all cells from the repository root to refresh this in-progress snapshot. Counts include only `classified` outcomes; pending, skipped and error outcomes are reported separately. Empty categories and categories with fewer than three results are explicitly shown.

Detected examples use the saved best-period folded flux profile, including its original display clipping and phase shift. Alias flags are retained. `no_period` has no detected period: those examples use a **one-day reference fold for display only**, with the run’s NGTS flux preprocessing. Norton categories describe period detection, not stellar types.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import sqlite3

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.io import fits
from IPython.display import display
from classify_upsilont import prepare_lightcurve, source_groups, cache_tile
from classify_norton import prepare_flux
from init_manifest import ArchiveSession

DATA_DIR = Path("data").resolve()
# Missing tiles may be large. Downloads use a separate cache, never the runners' caches.
DOWNLOAD_MISSING = True
PREVIEW_CACHE = Path("preview_cache").resolve()
PREVIEW_TILE = None  # Auto-select, or set a field+tile such as "NG0445-3056A".
EXAMPLES_PER_CATEGORY = 3
SEED = 42
REFERENCE_PERIOD_DAYS = 1.0  # Display only for Norton no_period; not a detection.
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.2})

PIPELINE = 'norton'
TITLE = 'Norton'


In [ ]:
output = DATA_DIR / "classifications" / PIPELINE
database = output / "checkpoints.sqlite"
if not database.is_file():
    raise FileNotFoundError(f"No checkpoints found: {database}. Run the classifier first.")
# One short, read-only transaction; release it before plotting/downloading.
with sqlite3.connect(database.as_uri() + "?mode=ro", uri=True, timeout=30) as db:
    db.execute("BEGIN")
    config = json.loads(db.execute("SELECT value FROM settings WHERE key='config'").fetchone()[0])
    records = db.execute("SELECT dp_id, source_id, status, payload FROM results ORDER BY dp_id, source_id").fetchall()
    completed_tiles = db.execute("SELECT COUNT(*) FROM tiles").fetchone()[0]
snapshot_time = datetime.now(timezone.utc).isoformat(timespec="seconds")
results = pd.DataFrame([
    {"dp_id": dp, "source_id": sid, "status": status, "payload": json.loads(payload)}
    for dp, sid, status, payload in records
], columns=["dp_id", "source_id", "status", "payload"])
results["label"] = results.payload.map(lambda p: p.get("label"))
manifest = pd.read_csv(DATA_DIR / "tile_manifest.csv", keep_default_na=False)
results = results.merge(manifest[["dp_id", "field", "tile", "filename", "cache_path", "download_url"]],
                        on="dp_id", how="left", validate="many_to_one")
classified = results.loc[results.status.eq("classified")].copy()
categories = config["classes"] if PIPELINE == "upsilon-t" else ["periodic_candidate", "alias_only", "no_period"]
categories = list(dict.fromkeys(categories + sorted(classified.label.dropna().unique().tolist())))
counts = classified.label.value_counts().reindex(categories, fill_value=0).rename("count").to_frame()
counts["percent_of_classified"] = (100 * counts["count"] / max(len(classified), 1)).round(2)
print(f"Snapshot: {snapshot_time} | {completed_tiles:,}/{len(manifest):,} completed tiles")
print(f"{len(results):,} checkpointed source/tile records; {len(classified):,} classified. Fields are not deduplicated.")
display(results.status.value_counts().rename_axis("status").to_frame("count"))
display(counts)
fig, ax = plt.subplots(figsize=(11, max(3, 0.32 * len(categories))))
bars = ax.barh(counts.index, counts["count"], color="steelblue")
ax.bar_label(bars, padding=3, fmt="%d")
ax.invert_yaxis()
ax.set(xlabel="Classified source/tile records", title=f"{TITLE}: classifications in progress")
ax.margins(x=0.15)
plt.tight_layout()
plt.show()


## Three examples per category from one tile

All examples in this notebook come from a single tile. Automatic selection prefers a locally cached tile, then maximizes category coverage and the number of examples (up to three per category). Set `PREVIEW_TILE` to a field+tile identifier to choose explicitly; use the same value in both notebooks to share that tile. The summary above still counts the full run. Categories absent or scarce in this tile have empty example panels; no other tiles are fetched to fill them.

Examples are sampled reproducibly (seed 42). At most one missing tile is downloaded into the shared `preview_cache/`. Set `DOWNLOAD_MISSING = False` for an offline preview. Checkpoints are read-only. Re-run all cells to refresh.

In [ ]:
def local_tile(row):
    paths = [PREVIEW_CACHE / "tiles" / row.filename,
             DATA_DIR / row.cache_path,
             DATA_DIR / "tiles" / "norton" / row.filename]
    return next((p for p in paths if p.is_file()), None)


def load_curves(selected):
    curves, failures = {}, {}
    # Open each tile only once and keep only the selected, preprocessed curves.
    for dp_id, group in selected.groupby("dp_id", sort=False):
        row = next(group.itertuples())
        try:
            path = local_tile(row)
            if path is None:
                if not DOWNLOAD_MISSING:
                    raise FileNotFoundError("Tile unavailable locally; enable DOWNLOAD_MISSING")
                record = {"field": row.field, "tile": row.tile,
                          "filename": row.filename, "cache_path": f"tiles/{row.filename}",
                          "download_url": row.download_url}
                with ArchiveSession() as session:
                    path = cache_tile(session, record, PREVIEW_CACHE)
            with fits.open(path, memmap=True, character_as_bytes=True) as hdus:
                data = hdus[1].data
                selections = dict(source_groups(data["SOURCE_ID"]))
                for source in group.itertuples():
                    key = (dp_id, source.source_id)
                    if source.source_id not in selections:
                        failures[key] = "Source absent from tile"
                        continue
                    prepare = prepare_lightcurve if PIPELINE == "upsilon-t" else prepare_flux
                    prepared, _, reason = prepare(data[selections[source.source_id]], config)
                    if reason:
                        failures[key] = reason
                    else:
                        curves[key] = prepared
        except (OSError, ValueError, RuntimeError) as exc:
            for source in group.itertuples():
                failures[(dp_id, source.source_id)] = str(exc)
    return curves, failures


# Prefer locally available photometry, then maximize category coverage in one tile.
def select_examples(classified, preview_tile=None):
    available = classified.copy()
    available["local"] = [local_tile(r) is not None for r in available.itertuples()]
    if preview_tile is not None:
        available = available.loc[(available.field + available.tile).eq(preview_tile)]
        if available.empty:
            raise ValueError(f"No classified results for PREVIEW_TILE={preview_tile!r}")
    if available.empty:
        return available, None
    coverage = available.groupby(["dp_id", "label"]).size().unstack(fill_value=0)
    ranking = pd.DataFrame({
        "categories": coverage.gt(0).sum(axis=1),
        "examples": coverage.clip(upper=EXAMPLES_PER_CATEGORY).sum(axis=1),
        "local": available.groupby("dp_id")["local"].any(),
    }).sort_values(["local", "categories", "examples"], ascending=False, kind="stable")
    chosen = ranking.index[0]
    pool = available.loc[available.dp_id.eq(chosen)]
    selected = pd.concat([
        pool.loc[pool.label.eq(category)].sample(frac=1, random_state=SEED)
        .head(EXAMPLES_PER_CATEGORY)
        for category in categories
    ], ignore_index=True)
    row = pool.iloc[0]
    return selected, row.field + row.tile


selected, selected_tile = select_examples(classified, PREVIEW_TILE)
print(f"Preview tile: {selected_tile or 'none (no classifications yet)'}; counts above cover the full run.")
if PIPELINE == "norton":
    needs_raw = selected.loc[selected.payload.map(
        lambda p: not (p.get("best_period") or {}).get("folded_profile")
    ).astype(bool)]
else:
    needs_raw = selected
curves, failures = load_curves(needs_raw)
display(selected[["label", "field", "tile", "source_id", "local"]])


In [ ]:
for category in categories:
    examples = selected.loc[selected.label.eq(category)]
    fig, axes = plt.subplots(1, EXAMPLES_PER_CATEGORY, figsize=(16, 4), squeeze=False)
    fig.suptitle(f"{category} — {counts.loc[category, 'count']:,} classified records across run; preview tile {selected_tile}")
    for i, ax in enumerate(axes[0]):
        if i >= len(examples):
            ax.text(0.5, 0.5, "No additional example in this tile", ha="center", va="center", transform=ax.transAxes)
            ax.set_axis_off()
            continue
        row = examples.iloc[i]
        payload = row.payload
        key = (row.dp_id, row.source_id)
        best = payload.get("best_period") or {}
        profile = best.get("folded_profile") if PIPELINE == "norton" else None
        title = f"{row.source_id}\n{row.field}{row.tile}"
        if profile:
            phase = np.asarray(profile["phase"], dtype=float)
            values = np.asarray(profile["flux"], dtype=float)
            errors = np.asarray(profile["flux_error"], dtype=float)
            period = best["period_days"]
            title += f"\nP={period:.6g} d; flag={best['period_flag']}"
            ax.set_ylabel("Mean flux (saved 100-bin profile)")
        elif key in curves:
            t, values, errors = curves[key]
            if PIPELINE == "upsilon-t":
                period = payload.get("features", {}).get("period", np.nan)
                title += f"\nP={period:.6g} d; probability={payload['probability']:.3f}"
                ax.set_ylabel("Relative magnitude")
                ax.invert_yaxis()
            else:
                t = t / 86400  # Norton preparation returns seconds.
                period = best.get("period_days", REFERENCE_PERIOD_DAYS)
                title += f"\nReference fold: {period:g} d (no detected period)"
                ax.set_ylabel("Flux")
            if not np.isfinite(period) or period <= 0:
                ax.text(0.5, 0.5, "No valid saved period", ha="center", transform=ax.transAxes)
                ax.set_title(title, fontsize=9)
                continue
            phase = (t / period) % 1
        else:
            ax.text(0.5, 0.5, failures.get(key, "Photometry unavailable"), ha="center", va="center",
                    wrap=True, fontsize=8, transform=ax.transAxes)
            ax.set_title(title, fontsize=9)
            continue
        valid = np.isfinite(phase) & np.isfinite(values) & np.isfinite(errors)
        phase, values, errors = phase[valid], values[valid], errors[valid]
        for cycle in (0, 1):
            ax.errorbar(phase + cycle, values, yerr=errors, fmt=".", ms=2,
                        elinewidth=0.4, alpha=0.45, color="tab:blue", rasterized=True)
        ax.set(xlabel="Phase (two cycles)", xlim=(0, 2), title=title)
        ax.title.set_fontsize(9)
    fig.tight_layout()
    plt.show()
    plt.close(fig)
if failures:
    display(pd.DataFrame([{"dp_id": key[0], "source_id": key[1], "reason": reason}
                          for key, reason in failures.items()]))
